# AI Multi-Agent Recruitment System

A practical AI recruitment system built with Python and Google Gemini.

The system analyzes a job description and a candidate CV, extracts structured information, compares skills and experience, calculates deterministic match scores, generates personalized interview questions, evaluates interview answers, and produces a final hiring recommendation.

## Key Features

- Job Description analysis
- PDF CV text extraction
- Structured CV analysis
- Skills matching and gap detection
- Deterministic weighted scoring
- Personalized interview question generation
- AI interview answer evaluation
- Final hiring recommendation
- Secure Gemini API key handling with Google Colab Secrets

## System Architecture

```text
            Job Description
                  |
                  v
          +----------------+
          | Job Analyzer   |
          +----------------+
                  |
                  v
          Structured Job Data
                  |
                  |
                  |                 PDF CV
                  |                   |
                  |                   v
                  |            +-------------+
                  |            | PDF Tool    |
                  |            +-------------+
                  |                   |
                  |                   v
                  |              CV Text
                  |                   |
                  |                   v
                  |            +-------------+
                  |            | CV Analyzer |
                  |            +-------------+
                  |                   |
                  |                   v
                  |          Structured CV Data
                  |                   |
                  +---------+---------+
                            |
                            v
                    +----------------+
                    | Matching Agent |
                    +----------------+
                            |
                            v
                  Matched / Missing Skills
                            |
                            v
                   +----------------+
                   | Scoring Engine |
                   +----------------+
                            |
                            v
                  CV Match Score + Decision
                            |
                            v
                  +------------------+
                  | Interview Agent  |
                  +------------------+
                            |
                            v
                Personalized Questions
                            |
                            v
                +----------------------+
                | Interview Evaluator  |
                +----------------------+
                            |
                            v
                  Interview Score
                            |
                            v
                +----------------------+
                | Final Hiring Report  |
                +----------------------+
                            |
                            v
                 Hiring Recommendation

## How It Works

1. The **Job Analyzer Agent** extracts required skills, experience, and education from the job description.
2. A Python PDF tool extracts text from the candidate's CV.
3. The **CV Analyzer Agent** converts the CV into structured data.
4. The **Matching Agent** identifies matched skills, missing skills, and education compatibility.
5. A deterministic **Scoring Engine** calculates Skills, Experience, and Education scores.
6. The system generates personalized interview questions based on the candidate profile and skill gaps.
7. The **Interview Evaluator Agent** scores the candidate's answers.
8. A final hiring report combines the CV match score and interview score.

## Scoring Logic

### CV Match Weights

- Skills: **40%**
- Experience: **35%**
- Education: **25%**

### Education Scores

- Exact match: **100**
- Related field: **75**
- Unrelated field: **30**

### Match Categories

- 80–100: **Strong Match**
- 70–79: **Good Match**
- 60–69: **Needs Review**
- Below 60: **Not Suitable**

### Interview Evaluation Weights

- Practical Thinking: **50%**
- Clarity: **25%**
- Depth: **25%**

## Tech Stack

- Python
- Google Gemini API
- Google Colab
- PyPDF
- Multi-Agent Architecture
- Structured JSON Outputs
- Deterministic Scoring Logic
- Weighted Evaluation
- PDF Text Extraction
- Interview Evaluation

## How to Run

1. Open the notebook in Google Colab.
2. Add your Gemini API key to Colab Secrets using the name:

   `GEMINI_API_KEY`

3. Enable notebook access for the secret.
4. Install the required libraries.
5. Run the notebook cells from top to bottom.
6. Upload a candidate CV in PDF format.
7. Provide the job description.
8. Run the recruitment workflow to generate:
   - CV analysis
   - Job analysis
   - Skill matching
   - Match score
   - Interview questions
   - Interview evaluation
   - Final hiring recommendation

> Never hard-code or commit your API key to GitHub.

In [ ]:
!pip install -q google-genai

from google import genai
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

MODEL = "gemini-3.5-flash-lite"

print("Recruitment AI ready ✅")

In [ ]:
# =========================================
# JOB ANALYZER AGENT
# =========================================

import json

def job_analyzer_agent(job_description):

    prompt = f"""
    أنت مسؤول عن تحليل وصف وظيفة.

    استخرج فقط المعلومات التالية:
    1. skills
    2. experience_years
    3. education

    أرجع النتيجة بصيغة JSON فقط وبنفس الشكل التالي:

    {{
      "skills": ["skill1", "skill2"],
      "experience_years": 1,
      "education": "field"
    }}

    لا تضف أي شرح خارج الـ JSON.

    وصف الوظيفة:
    {job_description}
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    result = json.loads(response.text)

    return result


job_description = """
Junior AI Engineer

Required Skills:
Python, GitHub, APIs, AI Agents

Experience:
1 year

Education:
Computer Science or related field
"""

job_data = job_analyzer_agent(job_description)

print(job_data)

In [ ]:
# =========================================
# CV ANALYZER AGENT
# =========================================

def cv_analyzer_agent(cv_text):

    prompt = f"""
    أنت مسؤول عن تحليل CV لمتقدم لوظيفة.

    استخرج فقط المعلومات التالية:
    1. skills
    2. experience_years
    3. education

    إذا لم يكن لدى الشخص خبرة عمل:
    experience_years = 0

    أرجع النتيجة بصيغة JSON فقط وبنفس الشكل:

    {{
      "skills": ["skill1", "skill2"],
      "experience_years": 0,
      "education": "field"
    }}

    لا تضف أي شرح خارج الـ JSON.

    CV:
    {cv_text}
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    result = json.loads(response.text)

    return result


cv_text = """
Name: Ahmed

Education:
Bachelor in Information Systems

Skills:
Python, GitHub, APIs

Experience:
No professional work experience.
"""

cv_data = cv_analyzer_agent(cv_text)

print(cv_data)

In [ ]:
# =========================================
# MATCHING AGENT
# =========================================

def matching_agent(job_data, real_cv_data):

    prompt = f"""
    أنت مسؤول عن مقارنة متطلبات وظيفة مع بيانات متقدم.

    بيانات الوظيفة:
    {job_data}

    بيانات المتقدم:
    {cv_data}

    المطلوب:

    1. حدد المهارات المطلوبة الموجودة لدى المتقدم.
    2. حدد المهارات المطلوبة الناقصة.
    3. قارن تخصص المتقدم بالتعليم المطلوب.

    education_match يجب أن يكون قيمة واحدة فقط:
    exact
    related
    unrelated

    أرجع JSON فقط بالشكل التالي:

    {{
      "matched_skills": ["skill1", "skill2"],
      "missing_skills": ["skill3"],
      "education_match": "exact",
      "education_reason": "short reason"
    }}

    لا تحسب أي scores.
    لا تضف أي كلام خارج JSON.
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    result = json.loads(response.text)

    return result


match_data = matching_agent(
    job_data,
    cv_data
)

print(match_data)

In [ ]:
# =========================================
# SCORING ENGINE
# =========================================

def calculate_scores(job_data, real_cv_data, match_data):

    # -------------------------
    # Skills Score
    # -------------------------

    required_skills = len(job_data["skills"])
    matched_skills = len(match_data["matched_skills"])

    if required_skills == 0:
        skills_score = 100
    else:
        skills_score = (
            matched_skills / required_skills
        ) * 100


    # -------------------------
    # Experience Score
    # -------------------------

    required_experience = job_data["experience_years"]
    candidate_experience = cv_data["experience_years"]

    if required_experience == 0:
        experience_score = 100
    else:
        experience_score = (
            candidate_experience / required_experience
        ) * 100

        experience_score = min(
            experience_score,
            100
        )


    # -------------------------
    # Education Score
    # -------------------------

    education_scores = {
        "exact": 100,
        "related": 75,
        "unrelated": 30
    }

    education_score = education_scores[
        match_data["education_match"]
    ]


    return {
        "skills_score": round(skills_score, 2),
        "experience_score": round(experience_score, 2),
        "education_score": round(education_score, 2)
    }


scores = calculate_scores(
    job_data,
    cv_data,
    match_data
)

print(scores)

In [ ]:
# =========================================
# FINAL SCORE + DECISION
# =========================================

def calculate_final_score(scores):

    final_score = (
        scores["skills_score"] * 0.40 +
        scores["experience_score"] * 0.35 +
        scores["education_score"] * 0.25
    )

    final_score = round(final_score, 2)

    if final_score >= 80:
        decision = "Strong Match"

    elif final_score >= 70:
        decision = "Good Match"

    elif final_score >= 60:
        decision = "Needs Review"

    else:
        decision = "Not Suitable"

    return {
        "final_score": final_score,
        "decision": decision
    }


final_result = calculate_final_score(scores)

print(final_result)

In [ ]:
# =========================================
# FULL RECRUITMENT WORKFLOW
# =========================================

def run_recruitment_system(job_description, cv_text):

    print("=== JOB ANALYSIS ===")
    job_data = job_analyzer_agent(job_description)
    print(job_data)

    print("\n=== CV ANALYSIS ===")
    cv_data = cv_analyzer_agent(cv_text)
    print(cv_data)

    print("\n=== MATCHING ===")
    match_data = matching_agent(
        job_data,
        cv_data
    )
    print(match_data)

    print("\n=== SCORES ===")
    scores = calculate_scores(
        job_data,
        cv_data,
        match_data
    )
    print(scores)

    print("\n=== FINAL RESULT ===")
    final_result = calculate_final_score(scores)
    print(final_result)

    return {
        "job_data": job_data,
        "cv_data": cv_data,
        "match_data": match_data,
        "scores": scores,
        "final_result": final_result
    }


result = run_recruitment_system(
    job_description,
    cv_text
)

In [ ]:
!pip install -q pypdf

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from pypdf import PdfReader

def extract_text_from_pdf(file_name):

    reader = PdfReader(file_name)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text


# غيّر الاسم ده لاسم ملف الـCV اللي رفعته
cv_pdf_text = extract_text_from_pdf("Sample_AI_Candidate_CV.pdf")

print(cv_pdf_text[:2000])

In [ ]:
# =========================================
# ANALYZE REAL PDF CV
# =========================================

real_cv_data = cv_analyzer_agent(cv_pdf_text)

print(real_cv_data)

In [ ]:
# =========================================
# INTERVIEW AGENT
# =========================================

def interview_agent(job_data, cv_data, match_data):

    prompt = f"""
    أنت مسؤول عن إجراء مقابلة تقنية لمتقدم لوظيفة.

    بيانات الوظيفة:
    {job_data}

    بيانات المتقدم:
    {cv_data}

    نتيجة المقارنة:
    {match_data}

    أنشئ 4 أسئلة مقابلة فقط.

    السؤال 1:
    سؤال تقني عن واحدة من أقوى مهارات المتقدم.

    السؤال 2:
    سؤال عن مهارة موجودة لدى المتقدم ومطلوبة في الوظيفة.

    السؤال 3:
    سؤال عن واحدة من المهارات الناقصة الموجودة في missing_skills
    لمعرفة هل لديه معرفة بها رغم أنها غير موجودة في الـ CV.

    السؤال 4:
    سؤال شخصي وعملي عن مشروع أو تجربة مذكورة في الـ CV.
    إذا لم توجد مشاريع أو تجارب واضحة، اسأله عن تجربة عملية استخدم فيها مهاراته.

    اجعل الأسئلة مناسبة لمستوى Junior.

    أرجع 4 أسئلة فقط بشكل مرقم.
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return response.text


interview_questions = interview_agent(
    job_data,
    real_cv_data,
    match_data
)

print(interview_questions)

In [ ]:
!pip install -q pypdf

In [ ]:
# =========================================
# INTERVIEW EVALUATOR AGENT
# =========================================

def interview_evaluator_agent(question, answer):

    prompt = f"""
    أنت مسؤول عن تقييم إجابة متقدم في مقابلة تقنية Junior.

    السؤال:
    {question}

    إجابة المتقدم:
    {answer}

    قيّم الإجابة من 10 في:

    practical_thinking
    clarity
    depth

    أرجع JSON فقط بالشكل التالي:

    {{
      "practical_thinking": 8,
      "clarity": 7,
      "depth": 6,
      "feedback": "short feedback"
    }}

    لا تضف أي كلام خارج JSON.
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return json.loads(response.text)


def calculate_interview_score(evaluation):

    score = (
        evaluation["practical_thinking"] * 0.50 +
        evaluation["clarity"] * 0.25 +
        evaluation["depth"] * 0.25
    )

    # Convert from /10 to /100
    return round(score * 10, 2)


# TEST

question = "لو API بيرجع Error، هتبدأ تحل المشكلة إزاي؟"

answer = """
هشوف الأول نوع الـ error والـ status code،
وبعدها أراجع البيانات اللي ببعتها للـ API،
وأضيف logging و error handling عشان أعرف المشكلة جاية منين.
"""

evaluation = interview_evaluator_agent(
    question,
    answer
)

interview_score = calculate_interview_score(evaluation)

print("Evaluation:", evaluation)
print("Interview Score:", interview_score)

In [ ]:
# =========================================
# FINAL HIRING REPORT
# =========================================

def final_hiring_report(cv_match_score, interview_score):

    overall_score = (
        cv_match_score * 0.50 +
        interview_score * 0.50
    )

    overall_score = round(overall_score, 2)

    if overall_score >= 80:
        recommendation = "Strong Hire"

    elif overall_score >= 70:
        recommendation = "Hire"

    elif overall_score >= 60:
        recommendation = "Further Review"

    else:
        recommendation = "Do Not Hire"

    return {
        "cv_match_score": cv_match_score,
        "interview_score": interview_score,
        "overall_score": overall_score,
        "recommendation": recommendation
    }


report = final_hiring_report(
    final_result["final_score"],
    interview_score
)

print(report)